In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import random
import json

from ast import literal_eval
from pathlib import Path

import numpy as np


############
# Parameters
############

bl_most_frequent = pd.read_csv('baseline_most_frequent_metrics.csv', index_col=0)

bl_uniform = pd.read_csv('baseline_uniform_metrics.csv', index_col=0)

bl_stratified = pd.read_csv('baseline_stratified_metrics.csv', index_col=0)

display(bl_most_frequent)
display(bl_uniform)
display(bl_stratified)

,f1_macro,f1,mcc,f1_samples
infiltrazione_sfinteri,0.475728,0.000000,0.0,NaN
infiltrazione_organi_extra,0.443299,0.000000,0.0,NaN
coinvolgimento_riflessione_peritoneale,0.460000,0.000000,0.0,NaN
coinvolgimento_fascia_mesorettale,0.379310,0.000000,0.0,NaN
depositi_tumorali,0.480769,0.000000,0.0,NaN
emvi_esteso,0.454545,0.000000,0.0,NaN
stadio_N,0.465347,0.930693,0.0,NaN
stadio_N1c,0.480769,0.000000,0.0,NaN
mrf,0.333333,0.000000,0.0,NaN
emvi,0.400000,0.000000,0.0,NaN


,f1_macro,f1,mcc,f1_samples
infiltrazione_sfinteri,0.313196,0.058824,-0.215918,NaN
infiltrazione_organi_extra,0.444118,0.300000,0.008538,NaN
coinvolgimento_riflessione_peritoneale,0.444994,0.270270,0.073568,NaN
coinvolgimento_fascia_mesorettale,0.441379,0.400000,-0.097345,NaN
depositi_tumorali,0.367273,0.121212,-0.021009,NaN
emvi_esteso,0.350376,0.157895,-0.182700,NaN
stadio_N,0.467105,0.684211,0.083949,NaN
stadio_N1c,0.367273,0.121212,-0.021009,NaN
mrf,0.406593,0.428571,-0.185695,NaN
emvi,0.378444,0.297872,-0.210090,NaN


,f1_macro,f1,mcc,f1_samples
infiltrazione_sfinteri,0.437500,0.000000,-0.123278,NaN
infiltrazione_organi_extra,0.379310,0.000000,-0.241121,NaN
coinvolgimento_riflessione_peritoneale,0.400000,0.000000,-0.198811,NaN
coinvolgimento_fascia_mesorettale,0.593985,0.473684,0.195388,NaN
depositi_tumorali,0.454545,0.000000,-0.090351,NaN
emvi_esteso,0.520993,0.210526,0.042640,NaN
stadio_N,0.486413,0.847826,-0.024656,NaN
stadio_N1c,0.454545,0.000000,-0.090351,NaN
mrf,0.458319,0.508475,-0.075378,NaN
emvi,0.613672,0.470588,0.229416,NaN


In [113]:
target_columns = AnnotationsReduced.model_fields.keys()

reg_fields = get_regression_fields(AnnotationsReduced)
cl_fields = get_classification_fields(AnnotationsReduced)
mc_fields = get_multiple_choice_fields(AnnotationsReduced)
bc_fields = get_binary_classification_fields(AnnotationsReduced)
label_to_id_map = create_label_to_id_map(AnnotationsReduced)

In [114]:
def save_actual_row(res_dict: dict, split: str, actual_row: pd.Series, label_to_id_map):
    for field in (cl_fields + bc_fields):
        label = actual_row[field]
        idx = label_to_id_map[field]['label_to_id'][label]
        res_dict[split]['actual'][field].append(idx)
    for field in mc_fields:
        l = literal_eval(actual_row[field])
        bits = labels_to_bits(l, label_to_id_map[field]['label_to_id'])
        res_dict[split]['actual'][field].append(bits)

In [115]:
def dummy_classification(y_train: np.ndarray | list,
                         strategy: str,
                         len_validation: int,
                         len_test: int,
                         random_state=RANDOM_STATE) -> tuple[list[int]]:
    x_train = np.zeros(len(y_train))
    classifier = DummyClassifier(strategy=strategy, random_state=random_state)
    classifier.fit(x_train, y_train)
    x_validation = np.zeros(len_validation)
    y_validation = classifier.predict(x_validation)
    x_test = np.zeros(len_test)
    y_test = classifier.predict(x_test)
    return y_validation.tolist(), y_test.tolist()

# Most frequent method

In [116]:
strategy = 'most_frequent'
len_validation = len(validation_data)
len_test = len(test_data)

In [117]:
results_most_frequent = {
    'train': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'validation': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'test': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'info':{
        'regression_fields': reg_fields,
        'classification_fields': cl_fields,
        'binary_classification_fields': bc_fields,
        'multiple_choice_fields': mc_fields,
        'normalization_stats': {},
        'label_to_id_map': label_to_id_map,
    }
}

for split in ['train', 'validation', 'test']:
    # Save actual labels
    for _, row in validation_data.iterrows():
        save_actual_row(results_most_frequent,
                        split,
                        row,
                        label_to_id_map)
    # Save dummy predictions
    for field in bc_fields + cl_fields:
        y_train = results_most_frequent['train']['actual'][field]
        pred_validation, pred_test = dummy_classification(
            y_train,
            strategy,
            len_validation,
            len_test
        )
        results_most_frequent['validation']['predicted'][field] = pred_validation
        results_most_frequent['test']['predicted'][field] = pred_test
    for field in mc_fields:
        matrix_train = np.array(results_most_frequent['train']['actual'][field])
        matrix_validation = []
        matrix_test = []
        for i in range(matrix_train.shape[1]):
            y_validation, y_test = dummy_classification(
                y_train=matrix_train[:, i],
                strategy=strategy,
                len_validation=len_validation,
                len_test=len_test
            )
            matrix_validation.append(y_validation)
            matrix_test.append(y_test)
        results_most_frequent['validation']['predicted'][field] = np.array(matrix_validation).T.tolist()
        results_most_frequent['test']['predicted'][field] = np.array(matrix_test).T.tolist()

# Uniform method

In [118]:
strategy = 'uniform'
len_validation = len(validation_data)
len_test = len(test_data)

In [119]:
results_uniform = {
    'train': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'validation': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'test': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'info':{
        'regression_fields': reg_fields,
        'classification_fields': cl_fields,
        'binary_classification_fields': bc_fields,
        'multiple_choice_fields': mc_fields,
        'normalization_stats': {},
        'label_to_id_map': label_to_id_map,
    }
}

for split in ['train', 'validation', 'test']:
    # Save actual labels
    for _, row in validation_data.iterrows():
        save_actual_row(results_uniform,
                        split,
                        row,
                        label_to_id_map)
    # Save dummy predictions
    for field in bc_fields + cl_fields:
        y_train = results_uniform['train']['actual'][field]
        pred_validation, pred_test = dummy_classification(
            y_train,
            strategy,
            len_validation,
            len_test
        )
        results_uniform['validation']['predicted'][field] = pred_validation
        results_uniform['test']['predicted'][field] = pred_test
    for field in mc_fields:
        matrix_train = np.array(results_uniform['train']['actual'][field])
        matrix_validation = []
        matrix_test = []
        for i in range(matrix_train.shape[1]):
            y_validation, y_test = dummy_classification(
                y_train=matrix_train[:, i],
                strategy=strategy,
                len_validation=len_validation,
                len_test=len_test
            )
            matrix_validation.append(y_validation)
            matrix_test.append(y_test)
        results_uniform['validation']['predicted'][field] = np.array(matrix_validation).T.tolist()
        results_uniform['test']['predicted'][field] = np.array(matrix_test).T.tolist()

# Stratified method

In [120]:
strategy = 'stratified'
len_validation = len(validation_data)
len_test = len(test_data)

In [121]:
results_stratified = {
    'train': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'validation': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'test': {
        'predicted': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)},
        'actual': {field: [] for field in (cl_fields + mc_fields + bc_fields + reg_fields)}
    },
    'info':{
        'regression_fields': reg_fields,
        'classification_fields': cl_fields,
        'binary_classification_fields': bc_fields,
        'multiple_choice_fields': mc_fields,
        'normalization_stats': {},
        'label_to_id_map': label_to_id_map,
    }
}

for split in ['train', 'validation', 'test']:
    # Save actual labels
    for _, row in validation_data.iterrows():
        save_actual_row(results_stratified,
                        split,
                        row,
                        label_to_id_map)
    # Save dummy predictions
    for field in bc_fields + cl_fields:
        y_train = results_stratified['train']['actual'][field]
        pred_validation, pred_test = dummy_classification(
            y_train,
            strategy,
            len_validation,
            len_test
        )
        results_stratified['validation']['predicted'][field] = pred_validation
        results_stratified['test']['predicted'][field] = pred_test
    for field in mc_fields:
        matrix_train = np.array(results_stratified['train']['actual'][field])
        matrix_validation = []
        matrix_test = []
        for i in range(matrix_train.shape[1]):
            y_validation, y_test = dummy_classification(
                y_train=matrix_train[:, i],
                strategy=strategy,
                len_validation=len_validation,
                len_test=len_test
            )
            matrix_validation.append(y_validation)
            matrix_test.append(y_test)
        results_stratified['validation']['predicted'][field] = np.array(matrix_validation).T.tolist()
        results_stratified['test']['predicted'][field] = np.array(matrix_test).T.tolist()

# Save results

In [122]:
results_most_frequent.pop('train')
results_uniform.pop('train')
results_stratified.pop('train')

{'predicted': {'morfologia': [],
  'riflessione_peritoneale_anteriore': [],
  'infiltrazione_tessuto_adiposo': [],
  'stadio_T': [],
  'posizione': [],
  'infiltrazione_organi_dettagli': [],
  'sedi_linfonodi': [],
  'infiltrazione_sfinteri': [],
  'infiltrazione_organi_extra': [],
  'coinvolgimento_riflessione_peritoneale': [],
  'coinvolgimento_fascia_mesorettale': [],
  'depositi_tumorali': [],
  'emvi_esteso': [],
  'stadio_N': [],
  'stadio_N1c': [],
  'mrf': [],
  'emvi': [],
  'metastasi': []},
 'actual': {'morfologia': [0,
   0,
   0,
   1,
   1,
   1,
   1,
   0,
   0,
   0,
   1,
   0,
   0,
   0,
   0,
   0,
   0,
   1,
   0,
   1,
   0,
   1,
   2,
   1,
   0,
   0,
   1,
   1,
   0,
   1,
   1,
   1,
   1,
   0,
   1,
   0,
   1,
   0,
   1,
   0,
   0,
   1,
   0,
   1,
   0,
   0,
   1,
   1,
   1,
   1,
   1,
   2,
   1,
   1],
  'riflessione_peritoneale_anteriore': [2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   2,
   1,
   1,
   2,
   0,

In [124]:
##############
# Save results
##############
with open('results_baseline_most_frequent.json', "w") as f:
    json.dump(results_most_frequent, f, indent=4)
    
with open('results_baseline_uniform.json', "w") as f:
    json.dump(results_uniform, f, indent=4)
    
with open('results_baseline_stratified.json', "w") as f:
    json.dump(results_stratified, f, indent=4)

print(f"Results saved")

Results saved
